# L4c: Shortest-Path Algorithms

A weighted graph turns movement between vertices into an optimization problem: among all feasible routes from a source to a target, which one has the smallest total edge weight? Shortest-path models answer this question for transportation, communication, logistics, and process-planning networks.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Formulate and update a shortest-path model:__ Define path cost in a weighted directed graph and distinguish a finite distance from an unreachable target. Use edge relaxation to update the distance and predecessor maps.
> * __Explain Dijkstra's greedy strategy:__ Trace the priority queue and settled set as the algorithm processes a graph. Use the nonnegative edge weights to explain why a settled distance is final and to state the cost of the computation.
> * __Explain Bellman–Ford's repeated relaxation:__ Trace the repeated edge-relaxation passes and relate their limit to the number of vertices. Distinguish a permissible negative edge from a reachable negative-weight cycle.

Let's get started!

___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including [`Include.jl`](Include.jl) and loading the resources used in this lecture.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the pinned course environment, loads the required packages, and makes the course functions available. For more information on the language used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

This lecture uses [the external `DataStructures.jl` package](https://juliacollections.github.io/DataStructures.jl/stable/), which is installed through the pinned course environment. Its [`PriorityQueue`](https://juliacollections.github.io/DataStructures.jl/stable/priority-queue/) type supports the Dijkstra implementation in [`ShortestPathAlgorithms.jl`](../../../code/src/ShortestPathAlgorithms.jl), which the local course package loads. [The `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/) supplies the checks in the worked comparison. The example graphs are constructed directly in the notebook, so there is no external data file to load.

___

## Shortest-Path Problem and Relaxation
Shortest-path models assign a cost to moving through a graph. Depending on the application, an edge weight might represent travel time, communication delay, monetary cost, or the number of unit operations in a chemical process. The common question is which sequence of feasible transitions has the smallest total weight.

> __Single-source shortest paths:__
>
> Let $\mathcal{G}=(\mathcal{V},\mathcal{E})$ be a directed graph with edge-weight function $w:\mathcal{E}\rightarrow\mathbb{R}$, and fix one source vertex $s\in\mathcal{V}$. A feasible route from $s$ to a target $t$ is an edge-following sequence written as:
> $$
> P=\langle v_0,v_1,\ldots,v_k\rangle,\qquad
> v_0=s,\quad v_k=t,\quad (v_i,v_{i+1})\in\mathcal{E}
> \text{ for }i=0,\ldots,k-1.
> $$
> The total weight accumulated along this route is given by:
> $$
> w(P):=\sum_{i=0}^{k-1}w(v_i,v_{i+1}).
> $$
> Let $\mathcal{P}_{s,t}$ denote the set of feasible routes from $s$ to $t$. Provided that no negative-weight cycle can be repeated on an $s$-to-$t$ route, the shortest-path distance is given by:
> $$
> d(s,t):=
> \begin{cases}
> \displaystyle\min_{P\in\mathcal{P}_{s,t}}w(P), & \mathcal{P}_{s,t}\neq\varnothing,\\[6pt]
> +\infty, & \mathcal{P}_{s,t}=\varnothing.
> \end{cases}
> $$
> Thus, $d(s,t)=+\infty$ identifies an unreachable target. When the optimum is finite, any selected shortest path $P^*$ is characterized by:
> $$
> P^*\in\arg\min_{P\in\mathcal{P}_{s,t}}w(P),\qquad w(P^*)=d(s,t).
> $$
> A single-source algorithm fixes $s$ and computes the distance map $\operatorname{dist}[v]=d(s,v)$ for all vertices $v\in\mathcal{V}$. It also records one predecessor for each reached vertex other than the source, for which $\operatorname{prev}[s]=\texttt{nothing}$. Following the recorded predecessors backward reconstructs a minimizing route; when several routes tie, the predecessor map may encode any one of them. If a reachable negative-weight cycle makes some route costs unbounded below, no finite single-source solution exists for the affected vertices, which is the failure Bellman–Ford detects.

Single-destination, single-pair, and all-pairs problems ask for different collections of these same quantities. In this lecture, both algorithms solve the single-source problem; we then select one target when we reconstruct a route. Enumerating every feasible path is usually impractical because the number of paths can grow exponentially. 

Efficient shortest-path algorithms instead exploit __optimal substructure__: if a shortest path from $s$ to $t$ passes through $u$, then its prefix from $s$ to $u$ must itself be a shortest path. Otherwise, replacing that prefix with a cheaper one would produce a cheaper path to $t$.

> __What does it mean to relax an edge?__
>
> Shortest-path algorithms avoid enumerating every route by maintaining two pieces of provisional state. The estimate $\operatorname{dist}[v]$ is the weight of the best source-to-$v$ route discovered so far, while $\operatorname{prev}[v]$ stores the preceding vertex on that route. At initialization, only the zero-edge route from $s$ to itself is known, so these maps are given by:
> $$
> \operatorname{dist}[v]=
> \begin{cases}
> 0, & v=s,\\
> +\infty, & v\neq s,
> \end{cases}
> \qquad \operatorname{prev}[v]=\texttt{nothing}.
> $$
> Now consider an edge $(u,v)\in\mathcal{E}$. If a finite route to $u$ has already been discovered, appending $(u,v)$ produces a candidate route to $v$ whose weight is given by:
> $$
> \operatorname{alt}(u,v)=\operatorname{dist}[u]+w(u,v).
> $$
> Relaxing $(u,v)$ means comparing this candidate with the current route to $v$ and updating both maps according to:
> $$
> \bigl(\operatorname{dist}[v],\operatorname{prev}[v]\bigr)\leftarrow
> \begin{cases}
> \bigl(\operatorname{alt}(u,v),u\bigr),
> & \operatorname{alt}(u,v)<\operatorname{dist}[v],\\[4pt]
> \bigl(\operatorname{dist}[v],\operatorname{prev}[v]\bigr),
> & \operatorname{alt}(u,v)\geq\operatorname{dist}[v].
> \end{cases}
> $$
> The strict comparison preserves the existing predecessor when the two routes tie. Whenever a finite shortest-path distance exists, every finite estimate is the weight of an actual discovered route and therefore satisfies the upper-bound invariant:
> $$
> d(s,v)\leq\operatorname{dist}[v].
> $$
> When finite shortest-path distances exist, immediately after $(u,v)$ is relaxed, the estimates also satisfy the local consistency condition $\operatorname{dist}[v]\leq\operatorname{dist}[u]+w(u,v)$. Repeated relaxation propagates cheaper routes across the graph; the global algorithm determines which edges are revisited and when these provisional estimates may be declared final.

Dijkstra and Bellman–Ford use this same local update differently. Dijkstra greedily finalizes the smallest tentative distance, which is safe only when every edge weight is nonnegative. Bellman–Ford repeatedly relaxes every edge, so a later negative edge may still improve an earlier estimate. The difference is not the update itself; it is the order and number of times the algorithms apply it.

___

## Dijkstra's Algorithm
Dijkstra's algorithm solves the single-source problem by repeatedly selecting the unsettled vertex with the smallest tentative distance. Its greedy step is valid when every edge weight is nonnegative.

__Initialization.__ Given $\mathcal{G}=(\mathcal{V},\mathcal{E})$, source $s\in\mathcal{V}$, and weights $w(u,v)\geq 0$, create distance and predecessor maps, an empty settled set $\mathcal{S}$, and a min-priority queue $\mathcal{Q}$. Set $\operatorname{dist}[s]=0$, set every other distance to $\infty$, set every predecessor to $\texttt{nothing}$, and insert $s$ into $\mathcal{Q}$ with priority zero.

__Iteration.__ While $\mathcal{Q}$ is not empty:

1. Remove a vertex $u$ having the smallest priority from $\mathcal{Q}$.
2. If $u\in\mathcal{S}$, skip it; otherwise add $u$ to $\mathcal{S}$. Its distance is now final.
3. For each outgoing edge $(u,v)$, compute $\operatorname{alt}=\operatorname{dist}[u]+w(u,v)$.
4. If $\operatorname{alt}<\operatorname{dist}[v]$, update $\operatorname{dist}[v]$, set $\operatorname{prev}[v]=u$, and assign $v$ priority $\operatorname{alt}$ in $\mathcal{Q}$.

> __Why is a settled distance final?__
>
> Suppose $u$ is the unsettled vertex with minimum tentative distance and assume, for contradiction, that a shorter path to $u$ exists. On that path, let $y$ be the first unsettled vertex and let $x$ be its settled predecessor. When $x$ was settled, relaxing $(x,y)$ produced a tentative distance no larger than the cost of the path prefix to $y$. Because all remaining edge weights are nonnegative, this value would be smaller than $\operatorname{dist}[u]$, contradicting the choice of $u$. The contradiction gives:
> $$
> u\in\arg\min_{v\notin\mathcal{S}}\operatorname{dist}[v]
> \quad\Longrightarrow\quad
> \operatorname{dist}[u]=d(s,u).
> $$
> This implication fails when a negative edge can reduce the cost later. The implementation rejects every graph containing a negative edge at its public interface. With adjacency lists and a priority queue, the running time is $\mathcal{O}((|\mathcal{V}|+|\mathcal{E}|)\log|\mathcal{V}|)$.

___

## Bellman–Ford Algorithm
Bellman–Ford removes Dijkstra's nonnegative-weight restriction by giving improvements time to propagate through the graph. Rather than finalize one vertex greedily, it repeatedly relaxes every edge.

__Initialization.__ Set $\operatorname{dist}[s]=0$, set every other distance to $\infty$, and set every predecessor to $\texttt{nothing}$.

__Relaxation passes.__ Repeat at most $|\mathcal{V}|-1$ times:

1. Visit every edge $(u,v)\in\mathcal{E}$.
2. If $\operatorname{dist}[u]$ is finite and $\operatorname{dist}[u]+w(u,v)<\operatorname{dist}[v]$, relax the edge and update $\operatorname{prev}[v]$.
3. If a complete pass makes no change, stop early because the distances have converged.

__Cycle check.__ Make one additional pass through the edges. If an edge from a finite-distance vertex can still be relaxed, report a reachable negative-weight cycle.

> __Why are $|\mathcal{V}|-1$ passes sufficient?__
>
> After pass $k$, Bellman–Ford has accounted for every source-to-vertex path containing at most $k$ edges. If no reachable negative-weight cycle exists, a shortest path can be chosen to be simple: repeating a vertex would only add a nonnegative cycle or a removable zero-cost cycle. A simple path visits at most $|\mathcal{V}|$ vertices and therefore uses at most $|\mathcal{V}|-1$ edges.
>
> The simple-path edge bound means that all finite shortest-path distances have propagated after at most $|\mathcal{V}|-1$ passes. If a reachable edge can still be relaxed on the next pass, some reachable cycle has negative total weight; traversing that cycle repeatedly drives path costs downward, so no finite shortest-path solution exists for affected vertices. A negative cycle disconnected from $s$ does not invalidate the single-source result.
>
> Bellman–Ford requires $\mathcal{O}(|\mathcal{V}|\,|\mathcal{E}|)$ time in the worst case and $\mathcal{O}(|\mathcal{V}|)$ storage for the distance and predecessor maps. The early-exit check can reduce the number of passes when estimates converge sooner.

### Choose the Algorithm from the Weight Contract

Dijkstra and Bellman–Ford return the same shortest-path distances when both algorithms' assumptions hold. Their predecessor maps may differ when several equal-cost shortest paths exist. Their edge-weight contracts and global relaxation schedules differ as follows:

| Question | Dijkstra | Bellman–Ford |
|:--|:--|:--|
| Edge weights | All weights are nonnegative | Negative edges are allowed |
| Global schedule | Settle the vertex with minimum tentative distance | Relax every edge repeatedly |
| Negative-cycle detection | Not supported | Detects a reachable negative-weight cycle |
| Worst-case time | $\mathcal{O}((\lvert\mathcal{V}\rvert+\lvert\mathcal{E}\rvert)\log\lvert\mathcal{V}\rvert)$ | $\mathcal{O}(\lvert\mathcal{V}\rvert\,\lvert\mathcal{E}\rvert)$ |
| Natural storage in the course implementation | Weighted adjacency list | Edge list |

Use Dijkstra when every edge weight is nonnegative and the priority-queue running time matters. Use Bellman–Ford when negative edges may occur or when the computation must detect a reachable negative-weight cycle.

___

## Worked Comparison
The two algorithms share the same distance and predecessor contracts, but they reach those results under different edge-weight assumptions. We will first compare them where both are valid, then isolate the behavior introduced by negative weights.

The graph below has two competing routes from source vertex 1 to target vertex 4. Because every edge weight is nonnegative, Dijkstra and Bellman–Ford should return the same distance map and reconstruct the same minimum-cost route.

In [ ]:
# Build the nonnegative directed graph -
# Each tuple has the form (source vertex, target vertex, edge weight).
nonnegative_edges = weighted_edges([
    (1, 2, 4.0), # direct route from source 1 to vertex 2
    (1, 3, 1.0), # inexpensive first step on the optimal route
    (3, 2, 2.0), # reaches vertex 2 for total cost 1 + 2 = 3, improving the direct cost 4
    (2, 4, 1.0), # completes the optimal route to target 4 for total cost 4
    (3, 4, 5.0), # alternative from vertex 3 reaches target 4 for total cost 6
])
source_vertex, target_vertex = 1, 4 # compute from vertex 1 and reconstruct the route ending at vertex 4

# Compute the single-source solution with both valid algorithms -
dijkstra_result = dijkstra(nonnegative_edges, source_vertex)      # distances and predecessors from greedy finalization
bellman_result = bellman_ford(nonnegative_edges, source_vertex)   # distances and predecessors from repeated relaxation

# Reconstruct each route from its predecessor map -
shortest_path = reconstruct_path(dijkstra_result.previous, source_vertex, target_vertex) # walk backward, then reverse
bellman_path = reconstruct_path(bellman_result.previous, source_vertex, target_vertex)    # independent route check

# Display the route, target cost, and independent agreement checks -
(
    path = shortest_path,                                                        # ordered source-to-target vertex ids
    cost = path_cost(nonnegative_edges, shortest_path),                          # sum the three selected edge weights
    distances_agree = dijkstra_result.distances == bellman_result.distances,     # compare every source-to-vertex cost
    paths_agree = shortest_path == bellman_path,                                 # compare the reconstructed target routes
)

The output confirms both algorithms obtain the route $1\rightarrow3\rightarrow2\rightarrow4$ with total cost $1+2+1=4$. Agreement is expected here because the nonnegative weights satisfy Dijkstra's precondition; Bellman–Ford provides an independent check using a different update schedule.

### Negative Edges and Reachable Negative Cycles
A negative edge does not by itself make the shortest-path problem ill-defined. In the first graph below, edge $(2,3)$ has weight $-2$, and Bellman–Ford finds the finite route $1\rightarrow2\rightarrow3\rightarrow4$ with cost $4-2+3=5$. Dijkstra rejects this graph because its greedy invariant no longer applies.

The second graph contains the reachable cycle $1\rightarrow2\rightarrow3\rightarrow1$ with total weight $1-2+0=-1$. Each trip around the cycle reduces the path weight by one, so Bellman–Ford must report that no finite shortest-path solution exists.

In [ ]:
# Build a graph with one negative edge but no negative-weight cycle -
# Bellman–Ford may use the negative edge because every source-to-target optimum remains finite.
negative_edge_graph = weighted_edges([
    (1, 2, 4.0),  # first step of the optimal route
    (1, 3, 5.0),  # direct alternative to vertex 3
    (2, 3, -2.0), # negative edge lowers the cost of reaching vertex 3 from 5 to 2
    (3, 4, 3.0),  # completes the optimal route to target 4 for total cost 5
])
negative_result = bellman_ford(negative_edge_graph, 1)              # no reachable negative cycle from source 1
negative_path = reconstruct_path(negative_result.previous, 1, 4)    # follow predecessors from target 4 back to source 1

# Build a reachable cycle with total weight 1 - 2 + 0 = -1 -
negative_cycle_graph = weighted_edges([
    (1, 2, 1.0),  # leave the source with cost 1
    (2, 3, -2.0), # reduce the running cost to -1
    (3, 1, 0.0),  # return to source 1 without offsetting the negative cost
])

# Capture expected errors so students can inspect them without stopping notebook execution -
dijkstra_rejection = try
    dijkstra(negative_edge_graph, 1) # must reject the -2 edge before running the greedy algorithm
    "unexpectedly accepted"         # sentinel reached only if the precondition check fails
catch error
    sprint(showerror, error)         # convert the expected ArgumentError into displayable text
end

cycle_rejection = try
    bellman_ford(negative_cycle_graph, 1) # an extra relaxation pass detects the reachable -1 cycle
    "unexpectedly accepted"              # sentinel reached only if cycle detection fails
catch error
    sprint(showerror, error)              # preserve the diagnostic while allowing later cells to run
end

# Display the valid Bellman–Ford route and both explicit rejections -
(
    path = negative_path,                                             # finite route 1 → 2 → 3 → 4
    cost = path_cost(negative_edge_graph, negative_path),             # 4 + (-2) + 3 = 5
    dijkstra_rejection = dijkstra_rejection,                          # negative-edge precondition message
    negative_cycle_rejection = cycle_rejection,                       # reachable-negative-cycle message
)

__What do we see?__ Bellman–Ford accepts the first graph and returns the expected finite path with cost 5. The two captured error messages separate the other cases: Dijkstra rejects the negative edge because its precondition is violated, while Bellman–Ford rejects the second graph because the source can reach a cycle whose total weight is negative. The executable checks turn these observations into contracts for the route, distance, algorithm agreement, and both rejection cases.

In [ ]:
# Check the numerical results and the public-interface contracts -
@testset "shortest-path contracts" begin
    # Nonnegative graph: route, cost, and both algorithms agree -
    @test shortest_path == [1, 3, 2, 4]                          # Dijkstra selects the three-edge minimum-cost route
    @test dijkstra_result.distances[4] == 4.0                    # target cost is 1 + 2 + 1
    @test dijkstra_result.distances == bellman_result.distances # both algorithms agree for every graph vertex
    @test shortest_path == bellman_path                          # the unique target route also agrees

    # Negative edge: Bellman–Ford still returns a finite optimum -
    @test negative_path == [1, 2, 3, 4]                          # the -2 edge belongs to the optimal route
    @test negative_result.distances[4] == 5.0                    # target cost is 4 - 2 + 3

    # Invalid inputs: each algorithm rejects the violated contract -
    @test_throws ArgumentError dijkstra(negative_edge_graph, 1)  # Dijkstra requires every edge weight to be nonnegative
    @test_throws ArgumentError bellman_ford(negative_cycle_graph, 1) # no finite optimum exists after a reachable -1 cycle
end

___

## Lab Exercises
In lab L4d, we will apply the existing shortest-path implementation to a weighted directed graph of alternative production routes. We will first predict the least-cost route by inspecting the process costs, then compute it with Dijkstra and independently verify the result with Bellman–Ford.

The lab closes with a sensitivity experiment: discounting one production step changes an edge weight and can change the optimal route. The algorithm-selection logic from this lecture remains visible throughout: the production costs are nonnegative, so Dijkstra is valid, while Bellman–Ford serves as a useful independent check.

___

## Summary
A shortest-path algorithm combines a local relaxation rule with a global schedule for deciding which edges to revisit and when a distance may be considered final.

> __Key Takeaways:__
>
> * __Relaxation connects local edges to global routes:__ Relaxing an edge tests whether extending a known route improves the current distance estimate for its target. The distance map stores the best costs found, while the predecessor map records the choices needed to reconstruct a route.
> * __Dijkstra finalizes distances greedily:__ When every edge weight is nonnegative, the unsettled vertex with the smallest tentative distance cannot be improved later. A priority queue lets the implementation select that vertex and update its outgoing neighbors efficiently.
> * __Bellman–Ford revisits edges to handle negative weights:__ Repeated relaxation propagates shortest paths containing successively more edges. An additional pass detects a reachable negative-weight cycle, for which no finite shortest-path solution exists for affected vertices.

In L4d, we apply these ideas to a production-planning network, verify the selected route with both algorithms, and test whether changing one process cost changes the optimum.

___